# 💪 FitBuddy — AI Workout Planner (Google Colab, no API keys)

Same architecture as the diagram, but **Gemini is replaced by open Hugging Face models that run locally in Colab**:

| Diagram block | This project |
|---|---|
| FastAPI Backend | `app/main.py` |
| HTML Templating (Jinja2 + CSS) | `app/templates/*.html` |
| Workout Logic (routes + AI) | `app/routes.py` |
| Admin Panel | `/admin/users` → `all_users.html` |
| Gemini 1.5 Pro (plan & updates) | `Qwen/Qwen2.5-1.5B-Instruct` |
| Gemini Flash (nutrition/recovery) | `Qwen/Qwen2.5-0.5B-Instruct` |
| AI Plan / Nutrition Generator | `app/updated_plan.py` |
| SQLite + SQLAlchemy | `app/database.py` → `fitbuddy.db` |

**Before you start:** `Runtime → Change runtime type → T4 GPU` (CPU works but each plan takes minutes).
Then run the cells top to bottom. No tokens, no keys, no ngrok account.

In [ ]:
!pip -q install fastapi "uvicorn[standard]" jinja2 sqlalchemy python-multipart transformers accelerate
!mkdir -p app/templates && touch app/__init__.py

In [ ]:
%%writefile app/database.py
from datetime import datetime
from sqlalchemy import create_engine, Column, Integer, Float, String, Text, DateTime, ForeignKey
from sqlalchemy.orm import declarative_base, sessionmaker, relationship

engine = create_engine("sqlite:///fitbuddy.db", connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(bind=engine, autoflush=False, autocommit=False)
Base = declarative_base()


class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True, index=True)
    name = Column(String(100), nullable=False)
    age = Column(Integer)
    weight = Column(Float)
    goal = Column(String(100))
    intensity = Column(String(50))
    created_at = Column(DateTime, default=datetime.utcnow)
    plans = relationship("WorkoutPlan", back_populates="user", cascade="all, delete-orphan")


class WorkoutPlan(Base):
    __tablename__ = "plans"
    id = Column(Integer, primary_key=True, index=True)
    user_id = Column(Integer, ForeignKey("users.id"))
    goal = Column(String(100))
    intensity = Column(String(50))
    original_plan = Column(Text)
    updated_plan = Column(Text)
    nutrition_tips = Column(Text)          # extra column to store the Flash-style tips
    created_at = Column(DateTime, default=datetime.utcnow)
    user = relationship("User", back_populates="plans")


def init_db():
    Base.metadata.create_all(engine)


def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

In [ ]:
%%writefile app/ai_service.py
"""Local Hugging Face inference. No API keys, no external services."""
import threading
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Open, ungated stand-ins for Gemini 1.5 Pro / Gemini Flash.
# On CPU-only runtimes, set both to "Qwen/Qwen2.5-0.5B-Instruct" for speed.
PLAN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # workout plan & updates
TIPS_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # nutrition / recovery tips

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_cache = {}
_lock = threading.Lock()


def _load(name):
    if name not in _cache:
        tok = AutoTokenizer.from_pretrained(name)
        dtype = torch.float16 if DEVICE == "cuda" else torch.float32
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(DEVICE)
        model.eval()
        _cache[name] = (tok, model)
    return _cache[name]


def generate(model_name, system, user, max_new_tokens=600, temperature=0.4):
    with _lock:  # one generation at a time keeps GPU memory predictable
        tok, model = _load(model_name)
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=0.9,
                repetition_penalty=1.05,
                pad_token_id=tok.eos_token_id,
            )
        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        return tok.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
%%writefile app/updated_plan.py
"""Builds prompts, calls the models, and parses replies into structured data."""
import re
from . import ai_service as ai

PLAN_SYSTEM = (
    "You are FitBuddy, a certified personal trainer. Give safe, practical, "
    "beginner-friendly advice. Follow the requested format exactly."
)
TIPS_SYSTEM = "You are FitBuddy, a sports nutritionist and recovery coach. Be brief and practical."


# ---------- generation ----------
def generate_plan(name, age, weight, goal, intensity):
    prompt = f"""Create a 5-day workout plan for {name} (age {age}, weight {weight} kg).
Goal: {goal}. Intensity: {intensity}.

Use EXACTLY this format and nothing else:
Day 1: <focus>
- <exercise> - <sets> x <reps>
- <exercise> - <sets> x <reps>
- <exercise> - <sets> x <reps>
Day 2: <focus>
...
Day 5: <focus>
Include a warm-up and cool-down line on each day."""
    return ai.generate(ai.PLAN_MODEL, PLAN_SYSTEM, prompt, max_new_tokens=750)


def revise_plan(current_plan, feedback):
    prompt = f"""Here is a workout plan:

{current_plan}

The user says: "{feedback}"

Rewrite the full plan so it respects this request. Keep the same format
(Day N: <focus> followed by "- exercise - sets x reps" lines). Output only the plan."""
    return ai.generate(ai.PLAN_MODEL, PLAN_SYSTEM, prompt, max_new_tokens=750)


def generate_tips(age, weight, goal, intensity):
    prompt = f"""Give nutrition and recovery tips for a {age}-year-old weighing {weight} kg
whose goal is "{goal}" with {intensity} training intensity.

Use EXACTLY this format:
Nutrition:
- tip
- tip
- tip
Recovery:
- tip
- tip
- tip"""
    return ai.generate(ai.TIPS_MODEL, TIPS_SYSTEM, prompt, max_new_tokens=300, temperature=0.5)


# ---------- parsing ----------
DAY_RE = re.compile(r"^\W*day\s*(\d+)\s*[:\-–]?\s*(.*)$", re.I)


def parse_plan(text):
    """Turn raw model text into [{title, focus, steps:[...]}]. Empty list if no days found."""
    days, cur = [], None
    for raw in (text or "").splitlines():
        line = raw.strip()
        if not line:
            continue
        m = DAY_RE.match(line)
        if m:
            cur = {"title": f"Day {m.group(1)}", "focus": m.group(2).strip(" *#:-"), "steps": []}
            days.append(cur)
        elif cur is not None:
            cur["steps"].append(line.lstrip("-*• ").strip())
    return days


def parse_tips(text):
    sections, current = {"Nutrition": [], "Recovery": []}, "Nutrition"
    for raw in (text or "").splitlines():
        line = raw.strip()
        if not line:
            continue
        low = line.lower().strip("*#: ")
        if len(low) < 25 and low.startswith("nutrition"):
            current = "Nutrition"
            continue
        if len(low) < 25 and low.startswith("recovery"):
            current = "Recovery"
            continue
        sections[current].append(line.lstrip("-*•0123456789. ").strip())
    return sections

In [ ]:
%%writefile app/routes.py
from pathlib import Path
from fastapi import APIRouter, Depends, Form, HTTPException, Request
from fastapi.responses import RedirectResponse
from fastapi.templating import Jinja2Templates
from sqlalchemy.orm import Session

from .database import get_db, User, WorkoutPlan
from . import updated_plan as ai_plan

router = APIRouter()
templates = Jinja2Templates(directory=str(Path(__file__).parent / "templates"))


@router.get("/")
def index(request: Request):
    return templates.TemplateResponse(request, "index.html", {})


@router.post("/generate")
def generate(
    name: str = Form(...),
    age: int = Form(...),
    weight: float = Form(...),
    goal: str = Form(...),
    intensity: str = Form(...),
    db: Session = Depends(get_db),
):
    user = User(name=name, age=age, weight=weight, goal=goal, intensity=intensity)
    db.add(user)
    db.commit()
    db.refresh(user)

    plan_text = ai_plan.generate_plan(name, age, weight, goal, intensity)
    tips_text = ai_plan.generate_tips(age, weight, goal, intensity)

    plan = WorkoutPlan(user_id=user.id, goal=goal, intensity=intensity,
                       original_plan=plan_text, nutrition_tips=tips_text)
    db.add(plan)
    db.commit()
    db.refresh(plan)
    return RedirectResponse(f"/plan/{plan.id}", status_code=303)


@router.get("/plan/{plan_id}")
def show_plan(plan_id: int, request: Request, db: Session = Depends(get_db)):
    plan = db.get(WorkoutPlan, plan_id)
    if not plan:
        raise HTTPException(404, "Plan not found")
    current = plan.updated_plan or plan.original_plan
    ctx = {
        "plan": plan,
        "user": plan.user,
        "days": ai_plan.parse_plan(current),
        "current_raw": current,
        "original_days": ai_plan.parse_plan(plan.original_plan),
        "tips": ai_plan.parse_tips(plan.nutrition_tips),
    }
    return templates.TemplateResponse(request, "result.html", ctx)


@router.post("/plan/{plan_id}/update")
def update_plan(plan_id: int, feedback: str = Form(...), db: Session = Depends(get_db)):
    plan = db.get(WorkoutPlan, plan_id)
    if not plan:
        raise HTTPException(404, "Plan not found")
    current = plan.updated_plan or plan.original_plan
    plan.updated_plan = ai_plan.revise_plan(current, feedback)
    db.commit()
    return RedirectResponse(f"/plan/{plan.id}", status_code=303)


# ---------- Admin panel ----------
@router.get("/admin/users")
def all_users(request: Request, db: Session = Depends(get_db)):
    users = db.query(User).order_by(User.created_at.desc()).all()
    return templates.TemplateResponse(request, "all_users.html", {"users": users})


@router.post("/admin/users/{user_id}/delete")
def delete_user(user_id: int, db: Session = Depends(get_db)):
    user = db.get(User, user_id)
    if user:
        db.delete(user)
        db.commit()
    return RedirectResponse("/admin/users", status_code=303)

In [ ]:
%%writefile app/main.py
from fastapi import FastAPI
from .database import init_db
from .routes import router

init_db()
app = FastAPI(title="FitBuddy")
app.include_router(router)

In [ ]:
%%writefile app/templates/base.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>FitBuddy</title>
<style>
  :root { --bg:#f6f7fb; --card:#fff; --ink:#1f2340; --muted:#6b7090; --accent:#6c4de6; --line:#e3e5f0; }
  * { box-sizing: border-box; }
  body { margin:0; font-family: system-ui, sans-serif; background:var(--bg); color:var(--ink); }
  nav { background:var(--accent); padding:14px 24px; display:flex; gap:20px; align-items:center; }
  nav a { color:#fff; text-decoration:none; font-weight:600; }
  nav .brand { font-size:1.2rem; margin-right:auto; }
  main { max-width:980px; margin:24px auto; padding:0 16px; }
  .card { background:var(--card); border:1px solid var(--line); border-radius:12px; padding:18px 20px; margin-bottom:16px; }
  .grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(280px,1fr)); gap:14px; }
  .day h4 { margin:0 0 8px; color:var(--accent); }
  .day ul, .card ul { margin:0; padding-left:18px; } li { margin:4px 0; }
  .muted { color:var(--muted); font-size:.92rem; }
  label { display:block; margin:12px 0 4px; font-weight:600; }
  input, select, textarea { width:100%; padding:10px; border:1px solid var(--line); border-radius:8px; font:inherit; }
  button { background:var(--accent); color:#fff; border:0; padding:11px 20px; border-radius:8px; font:inherit; font-weight:600; cursor:pointer; margin-top:16px; }
  button.danger { background:#d64550; margin:0; padding:6px 12px; }
  button:disabled { opacity:.6; cursor:wait; }
  table { width:100%; border-collapse:collapse; } th,td { padding:9px 8px; border-bottom:1px solid var(--line); text-align:left; }
  pre.raw { white-space:pre-wrap; background:#fafbff; padding:12px; border-radius:8px; border:1px solid var(--line); }
  .two { display:grid; grid-template-columns:1fr 1fr; gap:14px; } @media(max-width:640px){ .two{grid-template-columns:1fr;} }
  .tip { border-left:4px solid #f0a13a; } .rec { border-left:4px solid #3aa37a; }
</style>
</head>
<body>
<nav>
  <span class="brand">💪 FitBuddy</span>
  <a href="/">New plan</a>
  <a href="/admin/users">Admin</a>
</nav>
<main>{% block content %}{% endblock %}</main>
<script>
  document.querySelectorAll("form[data-busy]").forEach(f => f.addEventListener("submit", () => {
    const b = f.querySelector("button"); b.disabled = true; b.textContent = "AI is thinking… (can take a minute)";
  }));
</script>
</body>
</html>

In [ ]:
%%writefile app/templates/index.html
{% extends "base.html" %}
{% block content %}
<div class="card">
  <h2>Get your personalised workout plan</h2>
  <p class="muted">Generated by open-source models running inside this Colab notebook.</p>
  <form method="post" action="/generate" data-busy>
    <label>Name</label><input name="name" required>
    <div class="two">
      <div><label>Age</label><input name="age" type="number" min="10" max="100" required></div>
      <div><label>Weight (kg)</label><input name="weight" type="number" step="0.1" min="20" max="300" required></div>
    </div>
    <label>Goal</label>
    <select name="goal">
      <option>Weight loss</option><option>Muscle gain</option><option>Strength</option>
      <option>Endurance</option><option>General fitness</option>
    </select>
    <label>Intensity</label>
    <select name="intensity"><option>Low</option><option selected>Medium</option><option>High</option></select>
    <button type="submit">Generate plan</button>
  </form>
</div>
{% endblock %}

In [ ]:
%%writefile app/templates/result.html
{% extends "base.html" %}
{% block content %}
{% macro render_days(days, raw) %}
  {% if days %}
  <div class="grid">
    {% for d in days %}
    <div class="card day">
      <h4>{{ d.title }}{% if d.focus %} · {{ d.focus }}{% endif %}</h4>
      <ul>{% for s in d.steps %}<li>{{ s }}</li>{% endfor %}</ul>
    </div>
    {% endfor %}
  </div>
  {% else %}
  <pre class="raw">{{ raw }}</pre>
  {% endif %}
{% endmacro %}

<div class="card">
  <h2>{{ user.name }}'s plan</h2>
  <p class="muted">Age {{ user.age }} · {{ user.weight }} kg · Goal: <b>{{ plan.goal }}</b> · Intensity: <b>{{ plan.intensity }}</b></p>
</div>

<h3>{% if plan.updated_plan %}Updated plan{% else %}Your plan{% endif %}</h3>
{{ render_days(days, current_raw) }}

{% if plan.updated_plan %}
<details class="card">
  <summary>Show original plan</summary>
  <div style="margin-top:12px">{{ render_days(original_days, plan.original_plan) }}</div>
</details>
{% endif %}

<div class="two">
  <div class="card tip"><h4>🥗 Nutrition</h4><ul>{% for t in tips["Nutrition"] %}<li>{{ t }}</li>{% endfor %}</ul></div>
  <div class="card rec"><h4>😴 Recovery</h4><ul>{% for t in tips["Recovery"] %}<li>{{ t }}</li>{% endfor %}</ul></div>
</div>

<div class="card">
  <h4>Adjust this plan</h4>
  <form method="post" action="/plan/{{ plan.id }}/update" data-busy>
    <textarea name="feedback" rows="3" required placeholder="e.g. My knee hurts, avoid jumping. I only have dumbbells."></textarea>
    <button type="submit">Update plan</button>
  </form>
</div>
<p class="muted">AI-generated suggestions, not medical advice. Check with a professional if you have health concerns.</p>
{% endblock %}

In [ ]:
%%writefile app/templates/all_users.html
{% extends "base.html" %}
{% block content %}
<div class="card">
  <h2>All users ({{ users|length }})</h2>
  {% if users %}
  <table>
    <tr><th>ID</th><th>Name</th><th>Age</th><th>Weight</th><th>Goal</th><th>Intensity</th><th>Joined</th><th>Plans</th><th></th></tr>
    {% for u in users %}
    <tr>
      <td>{{ u.id }}</td><td>{{ u.name }}</td><td>{{ u.age }}</td><td>{{ u.weight }}</td>
      <td>{{ u.goal }}</td><td>{{ u.intensity }}</td>
      <td>{{ u.created_at.strftime('%Y-%m-%d %H:%M') }}</td>
      <td>{% for p in u.plans %}<a href="/plan/{{ p.id }}">#{{ p.id }}</a> {% endfor %}</td>
      <td><form method="post" action="/admin/users/{{ u.id }}/delete" onsubmit="return confirm('Delete this user?')">
        <button class="danger" type="submit">Delete</button></form></td>
    </tr>
    {% endfor %}
  </table>
  {% else %}<p class="muted">No users yet.</p>{% endif %}
</div>
{% endblock %}

### Step 1 · (Optional) test the AI layer without the web UI
Downloads the models on first run (~4 GB total, one time).

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from app import ai_service, updated_plan as ai_plan

print("Running on:", ai_service.DEVICE)
raw = ai_plan.generate_plan("Sam", 28, 72, "Muscle gain", "Medium")
print(raw)
print(ai_plan.parse_plan(raw)[:1])

### Step 2 · Launch the web app
Opens inside the notebook via Colab's built-in proxy, so no ngrok or tokens are needed. Use the second line if you'd rather have a full browser tab.

In [ ]:
import threading, time, uvicorn
from app.main import app

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)

from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=900)
# output.serve_kernel_port_as_window(8000)   # <- alternative: open in a new tab

### Step 3 · Peek at the database

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect("fitbuddy.db")
display(pd.read_sql("SELECT * FROM users", con))
display(pd.read_sql("SELECT id, user_id, goal, intensity, substr(original_plan,1,80) AS original_plan, created_at FROM plans", con))
con.close()

### Notes
- **Change models:** edit `PLAN_MODEL` / `TIPS_MODEL` in `app/ai_service.py` (e.g. `Qwen/Qwen2.5-3B-Instruct` for better plans on a T4, or `HuggingFaceTB/SmolLM2-1.7B-Instruct`). After editing a `.py` file, use **Runtime → Restart session** and re-run.
- **Schema changes:** delete `fitbuddy.db` if you change the tables (`!rm fitbuddy.db`).
- **Persistence:** Colab wipes files when the runtime resets. Mount Drive and point `sqlite:///` at it if you want to keep data.